# Notebook 01: Exploratory Data Analysis & Preprocessing (FAOSTAT & 15-Year Climate)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fahadrahman444/scientific-research-and-methodologies/blob/main/colab_notebooks/01_Bangladesh_Weather_Crop_EDA_and_Preprocessing.ipynb)

**Research Topic:** A Data-Driven Analysis of 15-Year Weather Variability and Its Impact on Agricultural Crop Yield in Bangladesh  
**Primary Dataset:** Official UN FAOSTAT (`data/raw/FAOSTAT_data_en_10-12-2026.csv`) & Bangladesh Meteorological Variability  
**Course:** CSE 418: Scientific Research and Methodologies Lab (UITS)  
**Author:** Fahad Rahman  

---
## 1. Overview & Cloud Environment Setup
This notebook explores the empirical UN FAOSTAT rice production data (1976–2024, focusing on 2009–2024) paired with 15-year meteorological indicators across Bangladesh.


In [ ]:
# Step 1: Libraries and Setup
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
print('Dependencies loaded successfully.')


In [ ]:
# Step 2: Automated Dataset Loading (Colab-Ready)
# Primary: FAOSTAT Raw and Cleaned Datasets
FAO_RAW_LOCAL = '../data/raw/FAOSTAT_data_en_10-12-2026.csv'
FAO_RAW_URL = f'{BASE_RAW_GITHUB}/data/raw/FAOSTAT_data_en_10-12-2026.csv'
FAO_CLEAN_LOCAL = '../data/processed/faostat_bangladesh_rice_cleaned.csv'
FAO_CLEAN_URL = f'{BASE_RAW_GITHUB}/data/processed/faostat_bangladesh_rice_cleaned.csv'
INTEGRATED_LOCAL = '../data/processed/bangladesh_faostat_weather_integrated_15yr.csv'
INTEGRATED_URL = f'{BASE_RAW_GITHUB}/data/processed/bangladesh_faostat_weather_integrated_15yr.csv'

# Load FAOSTAT dataset
if os.path.exists(FAO_RAW_LOCAL):
    df_fao_raw = pd.read_csv(FAO_RAW_LOCAL)
    print(f'Loaded raw FAOSTAT dataset locally ({len(df_fao_raw)} rows).')
else:
    df_fao_raw = pd.read_csv(FAO_RAW_URL)
    print(f'Loaded raw FAOSTAT dataset from GitHub ({len(df_fao_raw)} rows).')

# Load 15-Year Integrated Dataset
if os.path.exists(INTEGRATED_LOCAL):
    df_integrated = pd.read_csv(INTEGRATED_LOCAL)
    print(f'Loaded 15-Year integrated dataset locally ({len(df_integrated)} rows).')
else:
    df_integrated = pd.read_csv(INTEGRATED_URL)
    print(f'Loaded 15-Year integrated dataset from GitHub ({len(df_integrated)} rows).')

display(df_integrated.head(6))


## 2. Long-Term UN FAOSTAT Historical Rice Trajectory (1976–2024)
Visualizing nearly 5 decades of official UN FAOSTAT data on Bangladesh rice yield (kg/ha), production (tonnes), and harvested area (hectares).


In [ ]:
# Step 3: Pivot and Plot 49-Year FAOSTAT Historical Trends
pvt_fao = df_fao_raw.pivot(index='Year', columns='Element', values='Value').reset_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# 1. Yield (kg/ha)
axes[0].plot(pvt_fao['Year'], pvt_fao['Yield'], color='#2ca02c', lw=2.5, marker='o', ms=4)
axes[0].axvspan(2009, 2024, color='orange', alpha=0.2, label='15-Year Focus Window (2009-2024)')
axes[0].set_title('Official UN FAOSTAT: Bangladesh Rice Yield Evolution (1976-2024)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Yield (kg/ha)')
axes[0].legend()

# 2. Production (Million Tonnes)
axes[1].plot(pvt_fao['Year'], pvt_fao['Production'] / 1e6, color='#1f77b4', lw=2.5, marker='s', ms=4)
axes[1].axvspan(2009, 2024, color='orange', alpha=0.2)
axes[1].set_title('Official UN FAOSTAT: Total National Rice Production (Million Tonnes)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Production (Million Tonnes)')

# 3. Harvested Area (Million Hectares)
axes[2].plot(pvt_fao['Year'], pvt_fao['Area harvested'] / 1e6, color='#9467bd', lw=2.5, marker='^', ms=4)
axes[2].axvspan(2009, 2024, color='orange', alpha=0.2)
axes[2].set_title('Official UN FAOSTAT: Harvested Paddy Area (Million Hectares)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Area (Million ha)')

plt.tight_layout()
plt.show()


## 3. 15-Year Weather Variability vs. FAOSTAT Rice Yield (2009–2024)
Analyzing the non-linear relationship between seasonal weather shocks (Heat Stress Days, SPI Drought, Rain Anomaly) and national crop yields.


In [ ]:
# Step 4: Weather Variability and Crop Yield Anomaly Overlay
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Subplot 1: Yield vs Heat Stress Days
axes[0, 0].bar(df_integrated['year'], df_integrated['heat_days'], color='#d62728', alpha=0.7)
axes[0, 0].set_title('Annual Heat Stress Days (Tmax >= 35°C)', fontweight='bold')
axes[0, 0].set_ylabel('Days Count')

# Subplot 2: 15-Year Rice Yield (MT/ha)
axes[0, 1].plot(df_integrated['year'], df_integrated['yield_mt_ha'], marker='o', color='#2ca02c', lw=2.5)
axes[0, 1].set_title('FAOSTAT Rice Yield Rate (MT/ha)', fontweight='bold')
axes[0, 1].set_ylabel('MT / ha')

# Subplot 3: Standardized Precipitation Index (SPI)
colors = ['#1f77b4' if v >= 0 else '#ff7f0e' for v in df_integrated['spi']]
axes[1, 0].bar(df_integrated['year'], df_integrated['spi'], color=colors, alpha=0.85)
axes[1, 0].axhline(0, color='black', lw=1)
axes[1, 0].set_title('Standardized Precipitation Index (SPI-3)', fontweight='bold')
axes[1, 0].set_ylabel('SPI Z-Score (Drought < 0 < Flood)')

# Subplot 4: De-trended Yield Anomaly (%)
axes[1, 1].bar(df_integrated['year'], df_integrated['yield_climate_anomaly_pct'], color='#9467bd', alpha=0.8)
axes[1, 1].axhline(0, color='black', lw=1)
axes[1, 1].set_title('De-trended Climate-Induced Yield Anomaly (%)', fontweight='bold')
axes[1, 1].set_ylabel('Anomaly (%)')

plt.tight_layout()
plt.show()


In [ ]:
# Step 5: Correlation Matrix Between Climate Variables and FAOSTAT Yield
corr_features = ['yield_kg_ha', 'yield_climate_anomaly_pct', 'tmax', 'tmin', 'temp_mean', 'rain', 'spi', 'heat_days', 'humidity', 'sunshine']
corr = df_integrated[corr_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, square=True)
plt.title('Pearson Correlation: Bangladesh Weather Indicators vs FAOSTAT Rice Yields', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
